In [1]:
import lsdstreamburn.lsdstreamburn as sb

In [ ]:
my_dem = sb.get_dem(OT_api_key_fname="my_OT_api_key.txt", 
                    source="SRTM30", 
                    lower_left=[25.42, 77.48], 
                    upper_right = [31.48, 82.08],
                    prefix = "Chamoli_test")

print(my_dem)

In [ ]:
# Getting basin outline using Hydroshed basin boundary from running GEE script

import geopandas as gpd
import pandas as pd
import json
from shapely.geometry import shape
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt

# Paths
csv_path = '../Chamoli_basin_outline_hydroshed.csv'
dem_path = 'Chamoli_SRTMGL1.tif'
output_plot = 'basin_outline_over_dem.png'

# Load basin outline
df = pd.read_csv(csv_path)
df['geometry'] = df['.geo'].apply(lambda x: shape(json.loads(x)))
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')

# Open DEM
with rasterio.open(dem_path) as src:
    fig, ax = plt.subplots(figsize=(10, 10))
    show(src, ax=ax, title="Basin outline over DEM", cmap='terrain')

    # Reproject if needed
    if gdf.crs != src.crs:
        gdf = gdf.to_crs(src.crs)

    gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=2)

    plt.tight_layout()
    plt.savefig(output_plot, dpi=300)


In [ ]:
# Getting water mask on GEE Python API
import ee

import sys
sys.path.append('../geeCenterline')
import geeCenterline as geec # Use this package to dilate water mask and remove noise and holes

ee.Authenticate()
ee.Initialize()

from shapely.geometry import mapping
geom = df['geometry'].iloc[0]  # shapely.geometry.Polygon
geom_geojson = mapping(geom)  # gives a dict with 'type' and 'coordinates'

aoi = ee.Geometry.Polygon(geom_geojson['coordinates'])
start_date = ee.Date("2021-01-01")
end_date = start_date.advance(2, 'month')

# Apply filters to select relevant images
colFilter = ee.Filter.And(
    ee.Filter.bounds(aoi),
    ee.Filter.date(start_date, end_date)
)
DW_collection = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filter(colFilter)
print("Available Images:", DW_collection.size().getInfo())

# Select only the 'label' band
labelCol = DW_collection.select('label')

# 1) Create a binary water mask (1 if water, 0 otherwise)
def binary_water_mask(img):
    return img.eq(0)  # Water is 1, non-water is 0

waterMasks = labelCol.map(binary_water_mask)

# 2) Sum of water masks to count how many times a pixel is water
countWater = waterMasks.sum()

# 3) Count of valid pixels (not masked)
countValid = labelCol.count()

# 4) Compute water fraction per pixel
waterFraction = countWater.divide(countValid).reproject(
            crs=labelCol.first().projection(), 
            scale=10  # Explicitly setting scale to match Dynamic world scale 
        )

# 5) Threshold > 0.75 (75% water presence)
waterMajority = waterFraction.gt(0.75).rename('majorityWater').reproject(
            crs=labelCol.first().projection(), 
            scale=10
        )

# Clip to AOI
waterMajorityClipped = waterMajority.clip(aoi)

# connect the water masks divided by small gaps. The radius is the half scale of the gap to be filled.
waterLS1 = geec.close(waterMajorityClipped, radius=1.5, kernelType='square')

# Obtain the river mask
# identify the river from other water masks. The second input is the minimum area that is considered as the river.
# The second input is the connectivity of pixels. 4 means 4-connectivity and 8 means 8-connectivity.
riverMask = waterMajorityClipped.gt(0)
DWwater = waterMajorityClipped.eq(1).updateMask(riverMask)
DWwater2 = geec.close(DWwater, radius=1.5)
riverPS = geec.noise_removal(DWwater2, 500, 8)

In [ ]:
# Download a big channel mask in batch using geemap tiling
import geemap

# Create a fishnet grid over the AOI
fishnet = geemap.fishnet(aoi, h_interval=2.0, v_interval=2.0, delta=1)

# Download tiles in parallel
geemap.download_ee_image_tiles_parallel(
    image=riverPS,
    features=fishnet,
    out_dir="water_mask",       # Output directory (relative or absolute path)
    scale=10,                   # Spatial resolution in meters
    crs="EPSG:4326",            # Output projection
    num_threads=2               # 1 or 2 to avoid data quota limitation on GEE
)

In [ ]:
# Merging tiles of water mask in fishnet
import rasterio
from rasterio.merge import merge
from pathlib import Path

# List all .tif tiles
tile_dir = Path("water_mask")
tif_files = list(tile_dir.glob("*.tif"))

# Open all files
src_files_to_mosaic = [rasterio.open(str(fp)) for fp in tif_files]

# Merge into a single mosaic
mosaic, out_transform = merge(src_files_to_mosaic)

# Use metadata from first tile
out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "driver": "GTiff"
})

# Save combined output
with rasterio.open("DW_mask.tif", "w", **out_meta) as dest:
    dest.write(mosaic)

# Close all open files
for src in src_files_to_mosaic:
    src.close()

In [ ]:
# Water mask merged from geemap fishnet might have very big white space, we need to clip it before.

import rioxarray as rxr

# Load DEM and water mask
dem = rxr.open_rasterio("Chamoli_SRTMGL1.tif", masked=True).squeeze()
mask = rxr.open_rasterio("DW_mask.tif", masked=True).squeeze()

print(f"DEM crs: {dem.rio.crs}")
print(f"Water mask crs: {mask.rio.crs}")

# Ensure both rasters have the same CRS
if dem.rio.crs != mask.rio.crs:
    mask = mask.rio.reproject(dem.rio.crs)

# Clip water mask by DEM extent
clipped_mask = mask.rio.clip_box(
    minx=dem.rio.bounds()[0],
    miny=dem.rio.bounds()[1],
    maxx=dem.rio.bounds()[2],
    maxy=dem.rio.bounds()[3]
)

# Optional: save clipped mask
clipped_mask.rio.to_raster("DW_mask_clipped.tif")

DEM crs: EPSG:4326
Water mask crs: EPSG:4326


In [ ]:
# Apply stream burn algorithm and extract channel network

# Define burn depths of water and sediment pixels
WATER_DEPTH = 30
SEDI_DEPTH = 0

# Use the DEM file generated from last cell
DEM_FNAME = 'Chamoli_SRTMGL1.tif'

# This channel mask comes from land cover classification from Dynamic World land cover product
CHANNEL_MASK_FNAME = 'DW_mask_clipped.tif' 

# Define a prefix for the file name of generated river network
LOCATION_YEAR = 'Chamoli2021'  

burned_dem_path = sb.burning_driver(DataDirectory = "./", 
                   dem_fname=DEM_FNAME, 
                   channel_mask_fname=CHANNEL_MASK_FNAME,
                   location_year = LOCATION_YEAR, 
                   burn_water_depth=WATER_DEPTH,
                   burn_sediment_depth=SEDI_DEPTH,
                   area_thresh=15000,
                   resolution=30,
                   dem_source='SRTM30')

print(burned_dem_path)

Loading the class map
Loading the DEM
The minimum value of the burned DEM is:  33.0
Available burned DEM:  ['./burned_dem/Chamoli2021_SRTM30_Burned30m_UTM.tif', './burned_dem/Chamoli2021_SRTM30_Burned30m.tif']
Chosen DEM to burn is:  Chamoli2021_SRTM30_Burned30m.tif
Good news, I found lsdtt-basic-metrics. Lets go!
You are converting a DEM, you need to give it a grid spacing, if not the default is 30m
You are converting a DEM, you need to give it a grid spacing, if not the default is 30m
The projections is:
GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]
And some extra projection information strings:
None
WGS 84
The initial epsg code is: epsg:4326
The centre of the raster is at:
(80.53131519122695, 29.190979736284884)
dem_data.width is:

In [5]:
burned_dem_prefix = burned_dem_path.split('/')[-1].split('.')[0] + '_UTM'

sb.plot_network(DataDirectory="./burned_dem/", DEM_prefix=burned_dem_prefix)

Not finished yet
I am loading the points from: ./burned_dem/Chamoli2021_SRTM30_Burned30m_UTM_CN.csv
The object file prefix is: Chamoli2021_SRTM30_Burned30m_UTM_CN
Loading your file from csv
done
I am getting the hillshade
Your colourbar will be located: None
xsize: 5162 and y size: 7543
NoData is: -9999.0
Yoyoyoyo the EPSG is :EPSG:32644
EPSG:32644
The EPSGString is: EPSG:32644
minimum values are x: 147047.0 and y: 2811497.0
I made the ticks.
x labels are: 
['200', '300', '400', '500', '600']
x locations are:
[200000.0, 300000.0, 400000.0, 500000.0, 600000.0]
y labels are: 
['2900', '3000', '3100', '3200', '3300', '3400']
y locations are:
[2900000.0, 3000000.0, 3100000.0, 3200000.0, 3300000.0, 3400000.0]
This colourmap is: gray


/users/qchen7/.conda/envs/lsdenv/lib/python3.11/site-packages/lsdviztools/lsdmapfigure/plottingraster.py:456: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(self.tick_x_labels)
/users/qchen7/.conda/envs/lsdenv/lib/python3.11/site-packages/lsdviztools/lsdmapfigure/plottingraster.py:457: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(self.tick_y_labels)
/users/qchen7/.conda/envs/lsdenv/lib/python3.11/site-packages/lsdviztools/lsdmapfigure/plottingraster.py:1626: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  sc = self.ax_list[0].scatter(easting,northing,s=point_scale, c= unicolor,cmap=this_colourmap,edgecolors='none', alpha = alpha,zorder=zorder, marker = marker)


The number of axes are: 1
Axes(0,0;1x1)
Axes(0,0;1x1)
Now I'll get some points
I am going to plot some points for you. The EPSG string is:EPSG:32644
pointtools GetUTMEastingNorthing, getting the epsg string: EPSG:32644
WARNING you must have a recent (>=6) version of proj and pyproj (>=2.4) for this to work 
EPSG:32644
I got the easting and northing
The data stream_order is not one of the data elements in this point data
The data stream_order is not one of the data elements in this point data
The data stream_order is not one of the data elements in this point data
I also got the data for scaling, which is in column stream_order
The size of the array is: 
(0,)
I am going to convert data to log for point scaling.
I am scaling your points for you
There doesn't seem to be any scaling data. Reverting to manual size.
I will plot the points now.
The colourmap is: gist_earth
I am only plotting the points.
The aspect ratio is: 0.6843430995625083
I need to adjust the spacing of the colourbar.
The